# 04 Data Cleaning

## Objective

Clean the raw Telco churn dataset so key fields are ready for downstream analysis and modeling. This notebook focuses on the known `TotalCharges` type issue, validates the cleaned result, and saves a cleaned dataset for later stages.


## Imports


In [7]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CLEANED_DATA_PATH, DATA_PATH
from src.data.data_cleaning import clean_total_charges
from src.data.data_loader import load_raw_data
from src.data.data_validation import validate_cleaned_data, validate_raw_data


## Load And Validate Raw Data


In [ ]:
df = load_raw_data()
validate_raw_data(df)
df_clean = df.copy()

print("Raw dataset loaded and validated successfully.")
print(f"Data source: {project_root / DATA_PATH}")
print(f"Shape: {df_clean.shape}")


Raw dataset loaded and validated successfully.
Data source: /Users/mohammadmubashir/VCode/Churn/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)


## Inspect Cleaning Targets


In [ ]:
blank_total_charges = df_clean['TotalCharges'].astype(str).str.strip().eq('')

print(f"TotalCharges dtype before cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Blank TotalCharges rows: {blank_total_charges.sum()}")

df_clean.loc[blank_total_charges, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(11)


TotalCharges dtype before cleaning: object
Blank TotalCharges rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


## Clean TotalCharges

The blank `TotalCharges` rows belong to customers with `tenure == 0`. Since `TotalCharges` represents the total amount billed so far, assigning `0` is logically consistent for new customers who have not completed any billing period yet. This keeps those records in the dataset instead of dropping them unnecessarily.


In [ ]:
df_clean = clean_total_charges(df_clean)

print(f"TotalCharges dtype after cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Missing TotalCharges after cleaning: {df_clean['TotalCharges'].isna().sum()}")
print(f"Rows with tenure == 0 and TotalCharges == 0: {((df_clean['tenure'] == 0) & (df_clean['TotalCharges'] == 0)).sum()}")


TotalCharges dtype after cleaning: float64
Missing TotalCharges after cleaning: 0
Rows with tenure == 0 and TotalCharges == 0: 11


In [11]:
df_clean["TotalCharges"].describe()


count    7043.000000
mean     2279.734304
std      2266.794470
min         0.000000
25%       398.550000
50%      1394.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64

## Records With Zero TotalCharges


In [ ]:
zero_total_charges = df_clean[df_clean['TotalCharges'] == 0]

print(f"Rows where TotalCharges is 0: {len(zero_total_charges)}")
display(zero_total_charges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])


Rows where TotalCharges is 0: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,0.0
753,3115-CZMZD,0,20.25,0.0
936,5709-LVOEQ,0,80.85,0.0
1082,4367-NUYAO,0,25.75,0.0
1340,1371-DWPAZ,0,56.05,0.0
3331,7644-OMVMY,0,19.85,0.0
3826,3213-VVOLG,0,25.35,0.0
4380,2520-SGTTA,0,20.00,0.0
5218,2923-ARZLG,0,19.70,0.0
6670,4075-WKNIU,0,73.35,0.0


## Validate Cleaned Dataset


In [ ]:
validate_cleaned_data(df_clean)

print("Cleaned dataset validated successfully.")
print(f"TotalCharges dtype: {df_clean['TotalCharges'].dtype}")
print(f"Missing values: {int(df_clean.isna().sum().sum())}")
print(f"Duplicate customerID values: {int(df_clean['customerID'].duplicated().sum())}")
print(f"Rows with tenure == 0 and TotalCharges == 0: {int(((df_clean['tenure'] == 0) & (df_clean['TotalCharges'] == 0)).sum())}")


Cleaned dataset validated successfully.
TotalCharges dtype: float64
Missing values: 0
Duplicate customerID values: 0
Rows with tenure == 0 and TotalCharges == 0: 11


## Cleaning Summary

| Step | Detail |
|---|---|
| Raw shape | (7043, 21) |
| Cleaned shape | (7043, 21) |
| Rows dropped | 0 |
| Dtype corrected | TotalCharges: object → float64 |
| Blanks handled | 11 blank strings filled with 0.0 |
| Nulls remaining | 0 |
| Saved to | data/interim/telco_churn_cleaned.csv |

### What notebook 05 receives
A clean dataset with TotalCharges as float64, no missing values, and no
duplicate rows. customerID is retained for traceability and will be dropped
before modeling in notebook 08. Encoding, scaling, and feature engineering
are deferred to notebook 08.


## Save Cleaned Data


In [14]:
output_path = project_root / CLEANED_DATA_PATH
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Saved shape: {df_clean.shape}")


Cleaned dataset saved to: /Users/mohammadmubashir/VCode/Churn/data/interim/telco_churn_cleaned.csv
Saved shape: (7043, 21)
